In [ ]:
"""
langchain: Framework para construir aplicações que utilizam modelos de linguagem (LLMs), facilitando a criação de fluxos complexos.

langchain_community: Extensão do LangChain que adiciona suporte a novos conectores e ferramentas da comunidade.

langchain-huggingface: Integração do LangChain com modelos da Hugging Face, permitindo uso simplificado de LLMs e embeddings.

langchainhub: Repositório de templates de prompts e componentes reutilizáveis para LangChain.

langchain_chroma: Integração do LangChain com ChromaDB, um banco de dados vetorial para armazenar e recuperar embeddings.

"""

'\nlangchain: Framework para construir aplicações que utilizam modelos de linguagem (LLMs), facilitando a criação de fluxos complexos.\n\nlangchain_community: Extensão do LangChain que adiciona suporte a novos conectores e ferramentas da comunidade.\n\nlangchain-huggingface: Integração do LangChain com modelos da Hugging Face, permitindo uso simplificado de LLMs e embeddings.\n\nlangchainhub: Repositório de templates de prompts e componentes reutilizáveis para LangChain.\n\nlangchain_chroma: Integração do LangChain com ChromaDB, um banco de dados vetorial para armazenar e recuperar embeddings.\n\n'

In [ ]:
"""
transformers: Biblioteca da Hugging Face para carregar, treinar e usar modelos de NLP, como GPT, BERT e LLaMA.

einops: Facilita operações avançadas de manipulação de tensores, tornando-as mais intuitivas e eficientes, especialmente em deep learning.

accelerate: Otimiza o uso de hardware (CPU/GPU) para modelos de machine learning, facilitando o treinamento e a inferência distribuída.

bitsandbytes: Implementa técnicas de quantização para reduzir o tamanho e o consumo de memória de modelos grandes, permitindo execução em hardwares mais limitados.
"""


'\ntransformers: Biblioteca da Hugging Face para carregar, treinar e usar modelos de NLP, como GPT, BERT e LLaMA.\n\neinops: Facilita operações avançadas de manipulação de tensores, tornando-as mais intuitivas e eficientes, especialmente em deep learning.\n\naccelerate: Otimiza o uso de hardware (CPU/GPU) para modelos de machine learning, facilitando o treinamento e a inferência distribuída.\n\nbitsandbytes: Implementa técnicas de quantização para reduzir o tamanho e o consumo de memória de modelos grandes, permitindo execução em hardwares mais limitados.\n'

In [ ]:
!pip install -q transformers einops accelerate bitsandbytes
!pip install -q langchain langchain_community langchain-huggingface langchainhub langchain_chroma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 58.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Ge

In [ ]:
import torch
import os
import getpass

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline

from langchain.prompts import PromptTemplate
from langchain_core.prompts import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
    MessagesPlaceholder,
)
from langchain_core.messages import SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
os.environ["HF_TOKEN"] = ""

In [ ]:

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quantization_config)
tokenizer = AutoTokenizer.from_pretrained(model_id)

pipe = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.1,
    max_new_tokens=700,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
)
llm = HuggingFacePipeline(pipeline=pipe)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now default to True since model is quantized.


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
template_rag = """
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
Você é um assistente virtual prestativo e está seguindo instruções gerais. Você tem experiência em análise de questões de olímpiadas, irei passar provas para sua analise,
Use os seguintes pedaços de contexto recuperado para atribuir disciplinas (pode ser mais de uma) a cada questão, embase sua escolha com argumentos bem fundamentados. Ao analisar cada prova, além da classificação, faça um levantamento de quantas questões englobaram cada disciplina.
Se você não sabe a resposta, apenas diga que não sabe. Mantenha a resposta concisa. Lembre-se, seu papel não é responder a pergunta e sim  analisar e atribuir a elas disciplinas.
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Pergunta: {pergunta}
Contexto: {contexto}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
"""

In [ ]:
prompt_rag = PromptTemplate.from_template(template_rag)
print(prompt_rag)

input_variables=['contexto', 'pergunta'] input_types={} partial_variables={} template='\n<|begin_of_text|>\n<|start_header_id|>system<|end_header_id|>\nVocê é um assistente virtual prestativo e está seguindo instruções gerais. Você tem experiência em análise de questões de olímpiadas, irei passar provas para sua analise,\nUse os seguintes pedaços de contexto recuperado para atribuir disciplinas (pode ser mais de uma) a cada questão, embase sua escolha com argumentos bem fundamentados. Ao analisar cada prova, além da classificação, faça um levantamento de quantas questões englobaram cada disciplina.\nSe você não sabe a resposta, apenas diga que não sabe. Mantenha a resposta concisa. Lembre-se, seu papel não é responder a pergunta e sim  analisar e atribuir a elas disciplinas.\n<|eot_id|>\n<|start_header_id|>user<|end_header_id|>\nPergunta: {pergunta}\nContexto: {contexto}\n<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>\n'


In [ ]:
!pip install langchain
!pip install pypdf2
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 7.1 MB/s eta 0:00:00


In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
def processar_pdf(path : str):
  loader = PyPDFLoader(path)
  docs = loader.load()
  return docs

In [ ]:
# loader = WebBaseLoader(web_paths = ("https://www.bbc.com/portuguese/articles/cd19vexw0y1o",),)
# docs = loader.load()
# loader = PyPDFLoader(pdf_path)
# docs = loader.load()



pdf_path = "/content/drive/MyDrive/PIBIC JR/questionsFeatures.pdf"
docs = processar_pdf(pdf_path)

In [ ]:
len(docs[0].page_content)

3196

In [ ]:
print(docs[0].page_content[:300])

História:  
As  questões  de  História  em  olimpíadas  geralmente  exigem  dos  participantes  a  capacidade  
de
 
contextualizar
 
eventos
 
históricos,
 
analisar
 
documentos
 
e
 
interpretar
 
diferentes
 
perspectivas
 
sobre
 
um
 
mesmo
 
fato.
 
A
 
13ª
 
Olimpíada
 
Nacional
 
em
 
Histó


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200, add_start_index = True)
splits = text_splitter.split_documents(docs)

In [ ]:
hf_embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-mpnet-base-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vectorstore = Chroma.from_documents(documents=splits, embedding=hf_embeddings)
retriever = vectorstore.as_retriever(search_type = "similarity", search_kwargs={"k": 3})

In [ ]:
prompt_rag = PromptTemplate(
    input_variables=["contexto", "pergunta"],
    template=template_rag,
)
prompt_rag

PromptTemplate(input_variables=['contexto', 'pergunta'], input_types={}, partial_variables={}, template='\n<|begin_of_text|>\n<|start_header_id|>system<|end_header_id|>\nVocê é um assistente virtual prestativo e está seguindo instruções gerais. Você tem experiência em análise de questões de olímpiadas, irei passar provas para sua analise,\nUse os seguintes pedaços de contexto recuperado para atribuir disciplinas (pode ser mais de uma) a cada questão, embase sua escolha com argumentos bem fundamentados. Ao analisar cada prova, além da classificação, faça um levantamento de quantas questões englobaram cada disciplina.\nSe você não sabe a resposta, apenas diga que não sabe. Mantenha a resposta concisa. Lembre-se, seu papel não é responder a pergunta e sim  analisar e atribuir a elas disciplinas.\n<|eot_id|>\n<|start_header_id|>user<|end_header_id|>\nPergunta: {pergunta}\nContexto: {contexto}\n<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>\n')

In [ ]:
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
chain_rag = ({"contexto": retriever | format_docs, "pergunta": RunnablePassthrough()}
             | prompt_rag
             | llm
             | StrOutputParser())

In [ ]:
caminho = "/content/drive/MyDrive/PIBIC JR/OCHE2019[1].pdf"
prova = processar_pdf(caminho)

print(len(prova[0].page_content))
prova_processada = """ """

for i in range(len(prova)):
  prova_processada += prova[i].page_content

#print(prova_processada[:500])



questions = prova_processada.split("QUESTÃO")

questions.pop(0)

print(questions[0])

630
 1  
IMAGEM 1 
 
Solenidade de Abertura do Encontro Internacional do Centenário do Eclipse, realizada na 
cidade de Sobral – Ce, em 29 de maio de 2019, com a presença do Governador do Ceará, 
Camilo Santana; do Ministro da Ciência, Tecnologia, Inovações e Comunicações, Marcos 
Pontes; do titular da Secretaria de Ciência e Tecnologia do Estado do Ceará - SECITECE, 
Inácio Arruda; e diversas autoridades estaduais e municipais. 
Fonte: https://www.sct.ce.gov.br/2019/05/29/teoria-da-relatividade-aberto-encontro-
internacional-do-centenario-do-eclipse-de-sobral/, acesso 04 ago. 2019 
 
IMAGEM 2Reinauguração do Museu do Eclipse em Sobral – Ce, em maio de 2019, em meio às 
comemorações pelo Centenário do Eclipse (1919 -2019). O Museu fica localizado na 
Praça do Patrocínio, local de observação do Eclipse há cem anos. 
Fonte: http://portal.iphan.gov.br/ce/noticias/detalhes/5103/museu-do-eclipse-reabre-as-
portas-em-sobral-ce-apos-obra-de-requalificacao, acesso 05 ago. 2019. 
 
IMAGEM 3 
 


In [ ]:
prompt = f"""
Prova que deve ter suas questões analisadas: {questions[0]} (a classificação não é exclusiva, pode haver mais de uma disciplina envolvida).
Quais as disciplinas essas perguntas possuem dentre história, geografia, sociologia, filosofia, artes e literatura ?

Siga a seguinte estrutura para sua resposta para cada uma das questões (não invente questões, analise somente as que estão presentes no documento, analise somente as questões que estão completas, pode haver algumas incompletas):

Numero da questão:

Disciplinas envolvidas:

Justificativa:

(não quero que você responda as questões, apenas analise-a como pedi.)

"""

chain_rag.invoke(prompt)



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


'Número da questão: 1\n\nDisciplinas envolvidas: História, Geografia\n\nJustificativa: A questão refere-se à escolha da cidade de Sobral para a observação do eclipse solar em 1919, o que envolve a contextualização histórica do evento e a análise da localização geográfica da cidade.\n\nNúmero da questão: 2\n\nDisciplinas envolvidas: História, Artes\n\nJustificativa: A questão refere-se à reinauguração do Museu do Eclipse em Sobral, o que envolve a contextualização histórica do evento e a análise da importância cultural e histórica do museu.\n\nNúmero da questão: 3\n\nDisciplinas envolvidas: História, Geografia\n\nJustificativa: A questão refere-se à localização geográfica da cidade de Sobral e a escolha da cidade para a observação do eclipse solar em 1919, o que envolve a análise da localização geográfica e a contextualização histórica do evento.\n\nNúmero da questão: 4\n\nDisciplinas envolvidas: História, Sociologia\n\nJustificativa: A questão refere-se à escolha da cidade de Sobral pa